# Install the frozen paper data from Zenodo

This is the **only paper notebook that accesses the network**. Run all cells
once before any experiment notebook. It downloads the two immutable archives
from [Zenodo record 22107490](https://zenodo.org/records/22107490), verifies
their published size and MD5 checksum, extracts paths beginning with
`notebooks/` at the repository root, and validates the 369 expected assets.

Downloads are resumable and kept under `tmp/zenodo-22107490` until validation
succeeds. Existing extracted files are preserved by default. If the final
manifest check reports a corrupted pre-existing file, set
`OVERWRITE_EXISTING = True` and run the installation cell again.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path, PurePosixPath

import requests


ZENODO_RECORD_ID = "22107490"
ZENODO_DOI = "10.5281/zenodo.22107490"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

EXPECTED_ARCHIVES = {
    "healpix-resample-paper-core-data-v1.zip": {
        "size": 55_982_080,
        "md5": "c5d9305352b2d2dc1f5391b106e1b566",
    },
    "healpix-resample-paper-esri-multipatch-v1.zip": {
        "size": 1_111_019_520,
        "md5": "893a76967a2e3b788b035425ad4cddbd",
    },
}

OVERWRITE_EXISTING = False
DELETE_ARCHIVES_AFTER_SUCCESS = True


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "healpix_resample").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the healpix-resample checkout above {current}.")


REPO_ROOT = find_repo_root()
DOWNLOAD_DIR = REPO_ROOT / "tmp" / f"zenodo-{ZENODO_RECORD_ID}"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
print("Download cache:", DOWNLOAD_DIR)
print("Dataset DOI:", ZENODO_DOI)


Repository: /home/jovyan/healpix-resample
Download cache: /home/jovyan/healpix-resample/tmp/zenodo-22107490
Dataset DOI: 10.5281/zenodo.22107490


In [2]:
def md5_file(path: Path, chunk_size=8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def published_files() -> dict[str, dict]:
    response = requests.get(ZENODO_API, timeout=60)
    response.raise_for_status()
    record = response.json()
    if record.get("doi") != ZENODO_DOI:
        raise RuntimeError(f"Unexpected Zenodo DOI: {record.get('doi')!r}")
    return {item["key"]: item for item in record["files"]}


def download_resumable(url: str, destination: Path, expected_size: int) -> None:
    partial = destination.with_suffix(destination.suffix + ".part")
    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"[download] already complete: {destination.name}")
        return

    offset = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        if offset and response.status_code != 206:
            print("[download] server did not resume; restarting the partial file")
            offset = 0
        mode = "ab" if offset and response.status_code == 206 else "wb"
        downloaded = offset
        last_report = 0.0
        with partial.open(mode) as stream:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue
                stream.write(chunk)
                downloaded += len(chunk)
                now = time.monotonic()
                if now - last_report >= 5:
                    print(
                        f"[download] {destination.name}: "
                        f"{downloaded / 1e6:.1f}/{expected_size / 1e6:.1f} MB",
                        flush=True,
                    )
                    last_report = now

    if partial.stat().st_size != expected_size:
        raise IOError(
            f"Incomplete download for {destination.name}: "
            f"{partial.stat().st_size} != {expected_size} bytes. Run the cell again to resume."
        )
    os.replace(partial, destination)


def archive_relative_path(name: str) -> PurePosixPath | None:
    """Keep repository paths while tolerating an optional wrapper directory."""
    clean = PurePosixPath(name.replace("\\", "/"))
    parts = clean.parts
    if "notebooks" in parts:
        return PurePosixPath(*parts[parts.index("notebooks"):])
    allowed_root_files = {
        "data_manifest.csv", "git_commit.txt", "DATA_LICENSE.md",
        "DATA_LICENSES.md",
        "ATTRIBUTION.md", "DATA_DEPOSIT.md", "LICENSE", "DATA_README.md",
    }
    if clean.name in allowed_root_files:
        return PurePosixPath(clean.name)
    return None


def _safe_target(destination: Path, relative: PurePosixPath) -> Path:
    destination = destination.resolve()
    target = (destination / Path(*relative.parts)).resolve()
    if destination != target and destination not in target.parents:
        raise RuntimeError(f"Unsafe archive member: {relative}")
    return target


def _write_member(source, target: Path, overwrite: bool) -> bool:
    if target.exists() and not overwrite:
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".zenodo-part")
    with source, temporary.open("wb") as output:
        shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
    os.replace(temporary, target)
    return True


def _safe_target(destination: Path, relative: PurePosixPath) -> Path:
    destination = destination.resolve()
    target = (destination / Path(*relative.parts)).resolve()
    if destination != target and destination not in target.parents:
        raise RuntimeError(f"Unsafe archive member: {relative}")
    return target


def _write_member(source, target: Path, overwrite: bool) -> bool:
    if target.exists() and not overwrite:
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".zenodo-part")
    with source, temporary.open("wb") as output:
        shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
    os.replace(temporary, target)
    return True


def safe_extract(archive: Path, destination: Path, overwrite=False) -> tuple[int, int]:
    """Extract either a real ZIP or a TAR carrying a historical .zip suffix."""
    written = skipped = 0
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            bad = zf.testzip()
            if bad:
                raise IOError(f"ZIP CRC check failed for {archive.name}: {bad}")
            for member in zf.infolist():
                relative = archive_relative_path(member.filename)
                if relative is None:
                    continue
                target = _safe_target(destination, relative)
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                if _write_member(zf.open(member), target, overwrite):
                    written += 1
                else:
                    skipped += 1
        return written, skipped

    if tarfile.is_tarfile(archive):
        with tarfile.open(archive, mode="r:*") as tf:
            for member in tf:
                relative = archive_relative_path(member.name)
                if relative is None:
                    continue
                target = _safe_target(destination, relative)
                if member.isdir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                if not member.isfile():
                    raise RuntimeError(f"Unsupported TAR member type: {member.name}")
                source = tf.extractfile(member)
                if source is None:
                    raise IOError(f"Could not read TAR member: {member.name}")
                if _write_member(source, target, overwrite):
                    written += 1
                else:
                    skipped += 1
        return written, skipped

    raise IOError(f"Unsupported archive format: {archive}")


def validate_manifest() -> None:
    command = [
        sys.executable,
        str(REPO_ROOT / "notebooks" / "build_data_manifest.py"),
        "--check",
        "--doi",
        ZENODO_DOI,
    ]
    print("[validate]", " ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)


def install_zenodo_data() -> None:
    remote = published_files()
    if set(remote) != set(EXPECTED_ARCHIVES):
        raise RuntimeError(
            "Zenodo file list differs from the version expected by this notebook:\n"
            f"published={sorted(remote)}\nexpected={sorted(EXPECTED_ARCHIVES)}"
        )

    local_archives = []
    for name, expected in EXPECTED_ARCHIVES.items():
        item = remote[name]
        remote_md5 = item["checksum"].removeprefix("md5:")
        if item["size"] != expected["size"] or remote_md5 != expected["md5"]:
            raise RuntimeError(f"Published metadata changed unexpectedly for {name}.")

        archive = DOWNLOAD_DIR / name
        download_resumable(item["links"]["self"], archive, expected["size"])
        actual_md5 = md5_file(archive)
        if actual_md5 != expected["md5"]:
            raise IOError(
                f"MD5 mismatch for {name}: {actual_md5} != {expected['md5']}. "
                "Delete the local archive and rerun this notebook."
            )
        print(f"[checksum] {name}: OK ({actual_md5})")
        local_archives.append(archive)

    for archive in local_archives:
        written, skipped = safe_extract(
            archive, REPO_ROOT, overwrite=OVERWRITE_EXISTING
        )
        print(f"[extract] {archive.name}: wrote {written}, preserved {skipped}")

    validate_manifest()
    print("\nReady: all publication notebooks can now run with OFFLINE=True.")

    if DELETE_ARCHIVES_AFTER_SUCCESS:
        for archive in local_archives:
            archive.unlink(missing_ok=True)
        print("Downloaded ZIP files removed after successful validation.")


In [3]:
install_zenodo_data()


[download] healpix-resample-paper-core-data-v1.zip: 8.4/56.0 MB
[checksum] healpix-resample-paper-core-data-v1.zip: OK (c5d9305352b2d2dc1f5391b106e1b566)
[download] healpix-resample-paper-esri-multipatch-v1.zip: 8.4/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 75.5/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 142.6/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 209.7/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 276.8/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 352.3/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 478.2/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 570.4/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 662.7/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 755.0/1111.0 MB
[download] healpix-resample-paper-esri-multipatch-v1.zip: 889.2/1111.0 MB
[download] healpix-resample-paper-e

CalledProcessError: Command '['/srv/conda/envs/notebook/bin/python', '/home/jovyan/healpix-resample/notebooks/build_data_manifest.py', '--check', '--doi', '10.5281/zenodo.22107490']' returned non-zero exit status 1.